Este notebook executa o balanceamento descrito na especificação: cada janela de
referência (`Goal`, `shot`) é pareada com uma janela de `Background`
do **mesmo tamanho exato**, pertencente ao mesmo `(game_id, half)`, com clips
consecutivos e `arousal_score` abaixo do limiar definido.

In [23]:
import pandas as pd
import numpy as np

from balanced_df import balanced_df_window

pd.set_option("display.max_colwidth", 80)

In [24]:
# lista com jogos para ignorar
SKIP_GAMES = [
    "ligue-1_2015-2016_2015-09-19-18-30-reims-1-1-paris-sg",
    "serie-a_2015-2016_2015-11-01-22-45-lazio-1-3-ac-milan",
    "epl_2014-2015_2015-05-17-18-00-manchester-united-1-1-arsenal",
    "epl_2016-2017_2016-10-29-17-00-tottenham-1-1-leicester",
    "uefa-champions-league_2014-2015_2015-05-12-21-45-bayern-munich-3-2-barcelona",
    "uefa-champions-league_2015-2016_2015-11-03-22-45-psv-2-0-wolfsburg",
    "uefa-champions-league_2015-2016_2015-11-04-22-45-maccabi-tel-aviv-1-3-fc-porto",
    "uefa-champions-league_2015-2016_2015-11-03-22-45-shakhtar-donetsk-4-0-malmo-ff",
    "laliga_2014-2015_2015-05-23-19-30-barcelona-2-2-dep.-la-coruna",
    "laliga_2015-2016_2015-09-12-17-00-espanyol-0-6-real-madrid",
    "laliga_2015-2016_2016-04-02-21-30-barcelona-1-2-real-madrid",
    "laliga_2016-2017_2017-03-04-18-15-eibar-1-4-real-madrid",
    "laliga_2014-2015_2015-04-28-21-00-barcelona-6-0-getafe",
    "laliga_2014-2015_2015-05-23-21-30-real-madrid-7-3-getafe",
    "laliga_2015-2016_2015-09-12-21-30-atl.-madrid-1-2-barcelona",
    "laliga_2014-2015_2015-04-29-21-00-real-madrid-3-0-almeria",
    "laliga_2015-2016_2015-11-08-18-00-barcelona-3-0-villarreal",
]

In [25]:
# datasets paths
datasets_path = [
    '/mnt/storage_C4/gaussian_football/data/labels/labels_bundesliga.csv',
    '/mnt/storage_C4/gaussian_football/data/labels/labels_epl.csv',
    '/mnt/storage_C4/gaussian_football/data/labels/labels_laliga.csv',
    '/mnt/storage_C4/gaussian_football/data/labels/labels_ligue-1.csv',
    '/mnt/storage_C4/gaussian_football/data/labels/labels_serie-a.csv',
    '/mnt/storage_C4/gaussian_football/data/labels/labels_ucl.csv',
]

#### Parâmetros do balanceamento

- `highlight_names`: labels tratados como destaque principal (ex: gol).
- `shot_names`: labels tratados como finalização (ex: chute a gol).
- `category_map`: nomes de categoria amigáveis usados no `summary_df` e na
  coluna `window_category` do resultado.
- `threshold`: limiar de `arousal_score` abaixo do qual um clip de
  `Background` é elegível para compor uma janela.
- `max_trials`: tentativas aleatórias de janela `Background` por janela de
  referência, antes de marcar o evento como sem par (`matched=False`).


In [26]:
ligas = ['bundesliga', 'epl', 'laliga', 'ligue-1', 'serie-a', 'ucl']

HIGHLIGHT_NAMES = ["goal"]
SHOT_NAMES = ["shot"]
CATEGORY_MAP = {
    "goal": "Goal",
    "shot": "Shot",
    "Background": "Background",
}
BACKGROUND_LABEL = "Background"
THRESHOLD = 0.5
MAX_TRIALS = 1000
RANDOM_STATE = 1

In [27]:
for idx_liga in range(len(ligas)):
    df = pd.read_csv(datasets_path[idx_liga])
    df = df[~df['game_id'].isin(SKIP_GAMES)]

    balanced_df, unused_df, summary_df = balanced_df_window(
        df,
        clip_col="clip_path",
        label_col="type",
        event_id_col="event_id",
        score_col="arousal_score",
        game_col="game_id",
        highlight_names=HIGHLIGHT_NAMES,
        shot_names=SHOT_NAMES,
        category_map=CATEGORY_MAP,
        background_label=BACKGROUND_LABEL,
        threshold=THRESHOLD,
        max_trials=MAX_TRIALS,
        random_state=RANDOM_STATE,
    )

    print(f"\n=== {ligas[idx_liga]} ===")
    print(summary_df.to_string(index=False))
    print(f"matched={summary_df.attrs['matched_windows']}  unmatched={summary_df.attrs['unmatched_windows']}")

    balanced_df.to_csv(f'/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/{ligas[idx_liga]}_used.csv', index=False)
    unused_df.to_csv(f'/mnt/storage_C4/gaussian_football/data/balanced_datasets/unused/{ligas[idx_liga]}_unused.csv', index=False)
    summary_df.to_csv(f'/mnt/storage_C4/gaussian_football/data/balanced_datasets/summary/{ligas[idx_liga]}_summary.csv', index=False)


=== bundesliga ===
  category  n_windows  n_clips
      Goal        164     1510
      Shot        951     8356
Background       1115     9866
     TOTAL       2230    19732
matched=1115  unmatched=0

=== epl ===
  category  n_windows  n_clips
      Goal        290     2632
      Shot       1670    14413
Background       1960    17045
     TOTAL       3920    34090
matched=1960  unmatched=0

=== laliga ===
  category  n_windows  n_clips
      Goal        410     3714
      Shot       2089    18231
Background       2499    21945
     TOTAL       4998    43890
matched=2499  unmatched=0

=== ligue-1 ===
  category  n_windows  n_clips
      Goal        118     1054
      Shot        582     5106
Background        700     6160
     TOTAL       1400    12320
matched=700  unmatched=0

=== serie-a ===
  category  n_windows  n_clips
      Goal        304     2755
      Shot       1985    17309
Background       2289    20064
     TOTAL       4578    40128
matched=2289  unmatched=0

=== ucl ===
